<a href="https://colab.research.google.com/github/djamil13/Postdoc-Medical-AI-Cchallenge/blob/main/task3_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
TASK 3: Semantic Image Retrieval System
# task3_retrieval/
# ├── extract_embeddings.py
# ├── build_index.py
# ├── search.py
# └── evaluate_retrieval.py
Step 1: Extract Embeddings
python
# extract_embeddings.py
import torch
import numpy as np
from transformers import AutoModel, AutoImageProcessor
import faiss
class MedicalEmbeddingExtractor:
    def __init__(self, model_name="microsoft/BiomedVLP-CXR-BERT-specialized"):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                # For image embeddings
        self.image_processor = AutoImageProcessor.from_pretrained(model_name)
        self.image_model = AutoModel.from_pretrained(model_name).to(self.device)
        self.image_model.eval()
            def extract_image_embeddings(self, images, batch_size=32):
        """Extract embeddings for a list of images"""
               all_embeddings = []
                for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
                        # Convert numpy to tensor and preprocess
            batch_tensors = []
            for img in batch:
                if img.shape[0] == 1:
                    img = img[0]
                # Scale to 0-255 and convert to RGB
                img = (img * 255).astype(np.uint8)
                img = np.stack([img]*3, axis=-1)  # Convert to RGB
                batch_tensors.append(img)

            # Process with model's processor
            inputs = self.image_processor(images=batch_tensors, return_tensors="pt").to(self.device)
                        with torch.no_grad():
                outputs = self.image_model(**inputs)
                # Use [CLS] token or pooled output as embedding
                embeddings = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state[:, 0, :]
                all_embeddings.append(embeddings.cpu().numpy())
                return np.vstack(all_embeddings)
Step 2: Build FAISS Index
python
# build_index.py
def build_faiss_index(embeddings, dimension):
    """Build FAISS index for similarity search"""

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)
        # Create index
    index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity after normalization
        # Add embeddings
    index.add(embeddings)
        return index
def save_index(index, embeddings, metadata, path='retrieval_index'):
    """Save index and metadata"""
    faiss.write_index(index, f'{path}.faiss')
    np.save(f'{path}_embeddings.npy', embeddings)
    np.save(f'{path}_metadata.npy', metadata)
    print(f"Index saved to {path}.faiss")
Step 3: Image-to-Image Search
class MedicalImageRetriever:
    def __init__(self, index, embeddings, metadata, extractor):
        self.index = index
        self.embeddings = embeddings
        self.metadata = metadata  # Contains labels, image paths, etc.
        self.extractor = extractor
        def search_by_image(self, query_image, k=10):
        """Search similar images given a query image"""
                # Extract query embedding
        query_embedding = self.extractor.extract_image_embeddings([query_image])
                # Normalize
        faiss.normalize_L2(query_embedding)
                # Search
        distances, indices = self.index.search(query_embedding, k)
                results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            results.append({
                'rank': i+1,
                'index': idx,
                'similarity': float(dist),
                'label': self.metadata['labels'][idx],
                'image': self.metadata['images'][idx]  # Store image reference
            })
                return results
        def search_by_text(self, text_query, k=10):
        """Search images by text description"""
                # This requires a model that supports text encoding
        # For BiomedVLP, we can use the text encoder
        from transformers import AutoTokenizer
                tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedVLP-CXR-BERT-specialized")
                # Encode text
        inputs = tokenizer(text_query, return_tensors="pt", padding=True, truncation=True).to(self.extractor.device)
                with torch.no_grad():
            text_embeddings = self.extractor.image_model.get_text_features(**inputs)
            text_embeddings = text_embeddings.cpu().numpy()
                # Normalize
        faiss.normalize_L2(text_embeddings)
                # Search
        distances, indices = self.index.search(text_embeddings, k)
                results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            results.append({
                'rank': i+1,
                'index': idx,
                'similarity': float(dist),
                'label': self.metadata['labels'][idx],
                'image': self.metadata['images'][idx]
            })
                return results
        def visualize_results(self, query_image, results, query_type='image'):
        """Visualize query and retrieved images"""
                n_results = len(results)
        fig, axes = plt.subplots(1, n_results + 1, figsize=(4*(n_results+1), 4))
                # Show query
        if query_image.shape[0] == 1:
            query_display = query_image[0]
        else:
            query_display = query_image
                axes[0].imshow(query_display, cmap='gray')
        axes[0].set_title(f'Query\n({query_type})')
        axes[0].axis('off')
                # Show results
        for i, result in enumerate(results):
            img = result['image']
            if img.shape[0] == 1:
                img = img[0]
                        label_text = 'Pneumonia' if result['label'] == 1 else 'Normal'
            color = 'red' if result['label'] == 1 else 'green'
                        axes[i+1].imshow(img, cmap='gray')
            axes[i+1].set_title(f'Rank {result["rank"]}\n{label_text}\nSim: {result["similarity"]:.2f}',
                              color=color)
            axes[i+1].axis('off')

        plt.tight_layout()
        plt.savefig(f'retrieval_results_{query_type}.png')
        plt.show()
Step 4: Evaluate Retrieval Quality
def evaluate_precision_at_k(retriever, test_images, test_labels, k_values=[1, 3, 5, 10]):
    """Evaluate retrieval quality using Precision@k"""
        precision_at_k = {k: [] for k in k_values}
        for i, (query_image, query_label) in enumerate(zip(test_images, test_labels)):
        # Skip every 10th image to avoid too many queries
        if i % 10 != 0:
            continue
                    results = retriever.search_by_image(query_image, k=max(k_values))

        for k in k_values:
            retrieved_labels = [r['label'] for r in results[:k]]
            correct_count = sum(1 for label in retrieved_labels if label == query_label)
            precision = correct_count / k
            precision_at_k[k].append(precision)
        # Average precision
    avg_precision = {k: np.mean(precisions) for k, precisions in precision_at_k.items()}
        print("\n" + "="*60)
    print("📊 RETRIEVAL EVALUATION - PRECISION@K")
    print("="*60)
    for k in k_values:
        print(f"Precision@{k}: {avg_precision[k]:.4f}")
        # Clinical interpretation
    print("\n💡 CLINICAL INSIGHT:")
    if avg_precision[5] > 0.8:
        print("✅ High precision means similar cases (same diagnosis) are being grouped together.")
        print("   This system could help clinicians find similar prior cases for reference.")
    else:
        print("⚠️  Moderate precision indicates the embedding model may not capture")
        print("   all clinically relevant features. Consider fine-tuning on medical data.")
        return avg_precision

